# Plots

In [ ]:
library(readr)
library(purrr)
library(dplyr)
library(stringr)
library(fuzzyjoin)
library(ggplot2)
library(tidyverse)
library(Matrix)
library(reshape2)
library(RColorBrewer)
library(rstatix)
library(emmeans)
library(tibble)
library(tidyr)

In [ ]:
options(repr.matrix.max.cols = Inf,  # show all columns
        repr.matrix.max.rows = 200)  # adjust rows as you like

## 0. Plot parameters

In [ ]:
out_dir = "/ceph.groups/mshahbazi.grp/rsakata/Figures/ECAD_OE_IF/output"
filter_dapi = TRUE

analysis_summary_files = c(
"/ceph.groups/mshahbazi.grp/rsakata/EXP91/cellpose/output/plots/EXP91_analysis_summary.csv",
"/ceph.groups/mshahbazi.grp/rsakata/EXP92/all/cellpose/output/plots/EXP92_analysis_summary.csv",
"/ceph.groups/mshahbazi.grp/rsakata/EXP94/1_2/cellpose/output/plots/EXP94_analysis_summary.csv"
)

In [ ]:
# define const for visualization
FONT.SIZE <- 7
LABEL.FONT.SIZE <- 7
w <- 2 
h <- 2.5
LINE.W <- 0.5/2.141959

# Set geom defaults globally
update_geom_defaults("line",      list(linewidth = LINE.W))
update_geom_defaults("errorbar",  list(linewidth = LINE.W))
#update_geom_defaults("point",     list(size = LINE.W, stroke = LINE.W))

settheme <- theme_minimal() + 
  theme(
    text = element_text(family = "sans"), 
    panel.background = element_blank(),
    panel.grid.major = element_blank(), 
    panel.grid.minor = element_blank(),
    plot.background = element_blank(),
    axis.ticks = element_line(colour = "black", linewidth = LINE.W),
    axis.ticks.length = unit(0.1, "cm"), 
    axis.line = element_line(linewidth = LINE.W, colour = "black"),
    axis.title = element_text(size = FONT.SIZE),
    axis.text = element_text(colour = "black", size = FONT.SIZE),
    strip.text = element_text(size = FONT.SIZE), 
    strip.text.y.left = element_text(angle = 0, hjust = 1, size = FONT.SIZE),
    legend.position = "right",
    legend.title = element_text(size = FONT.SIZE), 
    legend.text = element_text(size = FONT.SIZE),
    legend.key.size = unit(0.3, "cm"),
    axis.text.x = element_text(colour = "black", angle = 0, size = LABEL.FONT.SIZE),
    title = element_text(size = FONT.SIZE) 
  )

In [ ]:
col_aneu = c("euploid"= "#D4D1B3","monosomy"="#109E9D","trisomy"="#F26B3B","complex"= "#886DB0")

col_condition_2 = c("control" = "#285F63", 
               "reversine" = "#CA4F33", 
               "mosaic"= "#E2A557")

col_condition = c("G_R"= "#5E5E5E","Grev_R"="#86AB30","Rrev_G"="#EB5951", "Grev_Rrev"="#F0A329")

col_GFP = c("TRUE"= "#86AB30","FALSE"="#8d8d8dff")

col_condition_3 = c("Developed" = "#285F62", 
               "Poor Quality" = "#CA4F33")



col_unspecified = "#5E5E5E"
col_GATA3 = "#489C9C"
col_NANOG = "#EA9542"
col_neg    = "#8d8d8dff"



## 1. Extract summary files

In [ ]:
# Read and combine all files into one dataframe
merged_df <- analysis_summary_files %>%
  map_dfr(read_csv)

In [ ]:
merged_df <- merged_df %>%
  mutate(
    sample_name = recode(
      sample_name,
      "rev_dox10_D2-D6" = "Mosaic_dox",
      "rev_none" = "Mosaic_none",
      "ctr_dox10_D2-D6" = "control_dox",
      "ctr_none" = "control_none"
    )
  )

In [ ]:
tbl <- merged_df %>%
  group_by( sample_name) %>%
  summarise(n_images = n_distinct(image), .groups = "drop") 
tbl

In [ ]:
merged_df = merged_df %>% filter(!sample_name %in% c("ctr_dox10_D0-D6", "rev_dox10_D0-D6"))

In [ ]:
tbl <- merged_df %>%
  group_by(exp, sample_name) %>%
  summarise(n = n_distinct(image), .groups = "drop") 
tbl

## 2. Preprocess

In [ ]:
colnames(merged_df)

# Filter cells with higher dapi levels
if (filter_dapi) {
  merged_df <- merged_df %>% 
    filter(Mean_dapi < DAPI_thresh)
}

In [ ]:
unique(merged_df$sample_name)

In [ ]:

order_sample <- c(
 'control_none', 'control_dox','Mosaic_none', 'Mosaic_dox')
merged_df <- merged_df %>%
  mutate(sample_name = factor(sample_name, levels = order_sample))

## Plot 

### E) %pos marker

In [ ]:
head(merged_df)

In [ ]:
unique(merged_df$condition)

In [ ]:
unique(merged_df$Treatment)

In [ ]:
unique(merged_df$treatment)

In [ ]:
plot_pct_bar_points <- function(
  data,                               # e.g., summary_df
  pct = pct_GATA3,                    # <-- column with % values to plot
  sample = sample_name,               # sample/category column
  condition = condition,              # grouping/fill column
  out_dir,                    # folder to save (optional)
  title = NULL,                       # default built from pct col name if NULL
  palette = NULL,                     # named vector for fill
  w = 3, h = 2,
  y_max = 110,
  y_ticks = 5,
  bar_width = 0.6,
  point_size = 0.3,
  point_alpha = 0.7,
  jitter_width = 0.05
) {
  pct      <- enquo(pct)
  sample   <- enquo(sample)
  condition<- enquo(condition)

  # default title from pct column name if not supplied
  if (is.null(title)) {
    title <- paste0(as_label(pct), "+")
  }

  # per-sample means (by condition) for the chosen pct column
  means_df <- data %>%
    group_by(!!sample, !!condition) %>%
    summarise(mean_pct = mean(!!pct, na.rm = TRUE), .groups = "drop")

  # build plot (reverse sample order, flip coords)
  p <- ggplot(data, aes(x = fct_rev(!!sample), y = !!pct)) +
    geom_col(
      data = means_df,
      aes(y = mean_pct, fill = !!condition),
      width = bar_width
    ) +
    geom_point(
      size = point_size, alpha = point_alpha,
      position = position_jitter(width = jitter_width),
      na.rm = TRUE
    ) +
    labs(x = "", y = "% positive", title = title, fill = rlang::as_name(condition)) +
    settheme +
    scale_y_continuous(limits = c(0, y_max), expand = c(0, 0),
                       breaks = scales::pretty_breaks(y_ticks)) +
    coord_flip()+
    theme(legend.position = "none")

  if (!is.null(palette)) {
    p <- p + scale_fill_manual(values = palette)
  }

  safe_title <- gsub("[^[:alnum:]_\\-]+","_", title)
  ggsave(file.path(out_dir, sprintf("%s.pdf", safe_title)),
                  plot = p, width = w, height = h)
  options(repr.plot.width=w, repr.plot.height=h)
  p
}

In [ ]:
summary_df <- merged_df %>%
  group_by(exp, image, sample_name) %>%
  summarise(
    n = n(),
    
    # Calculate simple percentages based on your existing boolean columns
    pct_GATA3 = 100 * mean(GATA3pos, na.rm = TRUE),
    pct_NANOG = 100 * mean(NANOGpos, na.rm = TRUE),
    pct_mcherry = 100 * mean(mcherrypos, na.rm = TRUE),
    #pct_GFP   = 100 * mean(GFPpos,   na.rm = TRUE),
    
    # Calculate negative cells (None of the markers are positive)
    pct_negative = 100 * mean(!( NANOGpos | GATA3pos), na.rm = TRUE),
    
    # Calculate double positives (Both markers are positive)
    pct_double_GATA3_NANOG = 100 * mean(GATA3pos & NANOGpos, na.rm = TRUE),
    
    .groups = "drop"
  )

# Plot % GATA3+
plot_pct_bar_points(summary_df, pct = pct_GATA3, condition = "sample_name", palette = col_condition, out_dir = out_dir, 
                    title = "E_pctGATA3+")
# Plot % NANOG+
plot_pct_bar_points(summary_df, pct = pct_NANOG, condition = "sample_name", palette = col_condition,out_dir = out_dir,
                    title = "E_pctNANOG+")
# Plot % mcherry+
plot_pct_bar_points(summary_df, pct = pct_mcherry, condition = "sample_name", palette = col_condition,out_dir = out_dir,
                    title = "E_pctmcherry+")
# Plot % neg
plot_pct_bar_points(summary_df, pct = pct_negative, condition = "sample_name", palette = col_condition,out_dir = out_dir, 
                    title = "E_pctnegative", y_max = 100)
# Plot % double+
plot_pct_bar_points(summary_df, pct = pct_double_GATA3_NANOG, condition = "sample_name", palette = col_condition,out_dir = out_dir, 
                    title = "E_pct_GATA3+_NANOG+", y_max = 100)



In [ ]:
## reformat for comparison with human embryos
plot_pct_bar_points <- function(
  data,                               
  pct = pct_GATA3,                    
  condition = condition,              
  out_dir,                    
  title = NULL,                       
  palette = "black",                     
  w = 2, h = 2,
  y_max = 110,
  y_ticks = 5,
  bar_width = 0.6,
  error_bar_width = 0.2,
  point_size = 0.5,
  point_alpha = 0.7,
  jitter_width = 0.1
) {
  pct       <- enquo(pct)
  condition <- enquo(condition)

  if (is.null(title)) {
    title <- paste0(as_label(pct), "+")
  }

  # 1. Calculate Summary Stats by CONDITION (Mean & SD)
  cond_summary <- data %>%
    group_by(!!condition) %>%
    summarise(
      mean_val = mean(!!pct, na.rm = TRUE),
      sd_val   = sd(!!pct, na.rm = TRUE),
      .groups  = "drop"
    )

  # 2. Build Plot
  # We map X to the Condition. 
  # Note: ensure your 'condition' column levels are set correctly before running this if you want specific order.
  p <- ggplot(data, aes(x = !!condition, y = !!pct)) +
    
    # A. The Bar (Mean of the condition)
    geom_col(
      data = cond_summary,
      aes(y = mean_val, fill = !!condition),
      width = bar_width,
      alpha = 0.6,           # Slight transparency to see points better
      show.legend = FALSE,
      fill = "grey"
    ) +
    
    # B. The Error Bars (Mean +/- SD)
    geom_errorbar(
      data = cond_summary,
      aes(
        y = mean_val, 
        ymin = pmax(0, mean_val - sd_val), # pmax(0, ...) prevents error bar going below 0
        ymax = mean_val + sd_val
      ),
      width = error_bar_width
    ) +
    
    # C. The Individual Points (Jittered)
    geom_jitter(
      size = point_size, 
      alpha = point_alpha,
      width = jitter_width,
      height = 0,             # Don't jitter vertically (keeps Y value accurate)
      na.rm = TRUE,
      color =  palette
    ) +
    
    labs(x = "", y = "%", title = title, fill = rlang::as_name(condition)) +
    settheme +
    scale_y_continuous(limits = c(0, y_max), expand = c(0, 0),
                       breaks = scales::pretty_breaks(y_ticks)) +
      #scale_fill_manual(values=col_condition)+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1),
      legend.position = "none"
    ) 

  safe_title <- gsub("[^[:alnum:]_\\-]+","_", title)
  ggsave(file.path(out_dir, sprintf("B_%s_by_condition.pdf", safe_title)),
                  plot = p, width = w, height = h)
  options(repr.plot.width=w, repr.plot.height=h)
  p
}

In [ ]:
summary_df = summary_df %>% filter(sample_name %in% c("Mosaic_none", "Mosaic_dox"))

In [ ]:
# Plot % double+
plot_pct_bar_points(summary_df , condition = sample_name, 
                    pct = pct_negative, palette = col_unspecified, out_dir = out_dir, 
                    title = "pct_negative", y_max = 100)

In [ ]:
check_test <- function(
  data,
  group_var = "sample_name",
  value_var = "pct_negative",
  conditions = c("D6_RrevG_D", "D6_RrevG_F"),
  alpha = 0.05
) {

  g <- data %>%
    filter(.data[[group_var]] %in% conditions) %>%
    split(.[[group_var]]) %>%
    lapply(function(d) na.omit(d[[value_var]]))

  g <- g[conditions]

  n1 <- length(g[[1]])
  n2 <- length(g[[2]])

  if (n1 < 3 || n2 < 3) {
    return(tibble(
      group1 = conditions[1],
      group2 = conditions[2],
      n1 = n1,
      n2 = n2,
      shapiro_p1 = NA_real_,
      shapiro_p2 = NA_real_,
      variance_p = NA_real_,
      normal = NA,
      recommended = "Too few points to assess normality"
    ))
  }

  sp1 <- tryCatch(shapiro.test(g[[1]])$p.value,
                  error = function(e) NA_real_)
  sp2 <- tryCatch(shapiro.test(g[[2]])$p.value,
                  error = function(e) NA_real_)

  normal <- all(!is.na(c(sp1, sp2))) &&
    sp1 > alpha &&
    sp2 > alpha

  variance_p <- tryCatch(
    var.test(g[[1]], g[[2]])$p.value,
    error = function(e) NA_real_
  )

  tibble(
    group1 = conditions[1],
    group2 = conditions[2],
    n1 = n1,
    n2 = n2,
    shapiro_p1 = sp1,
    shapiro_p2 = sp2,
    variance_p = variance_p,
    normal = normal,
    recommended = ifelse(
      normal,
      "Welch t-test",
      "Wilcoxon rank-sum test"
    )
  )
}

check_test(summary_df)

In [ ]:
wilcox_res <- summary_df %>%
  #filter(sample_name %in% c("D6_RrevG_D", "D6_RrevG_F")) %>%
  #mutate(sample_name = droplevels(sample_name)) %>%
  rstatix::wilcox_test(pct_negative ~ sample_name) %>%
  mutate(
    p_use = p,
    stars = case_when(
      p_use < 0.0001 ~ "****",
      p_use < 0.001  ~ "***",
      p_use < 0.01   ~ "**",
      p_use < 0.05   ~ "*",
      TRUE           ~ "ns"
    )
  )

wilcox_res

In [ ]:
# Plot % double+
plot_pct_bar_points(summary_df, condition = sample_name, 
                    pct = pct_GATA3, palette = col_GATA3, out_dir = out_dir, 
                    title = "pct_GATA3", y_max = 100)

In [ ]:
wilcox_res <- summary_df %>%
  #filter(sample_name %in% c("D6_RrevG_D", "D6_RrevG_F")) %>%
  #mutate(sample_name = droplevels(sample_name)) %>%
  rstatix::wilcox_test(pct_GATA3 ~ sample_name) %>%
  mutate(
    p_use = p,
    stars = case_when(
      p_use < 0.0001 ~ "****",
      p_use < 0.001  ~ "***",
      p_use < 0.01   ~ "**",
      p_use < 0.05   ~ "*",
      TRUE           ~ "ns"
    )
  )

wilcox_res

In [ ]:
# Plot % double+
plot_pct_bar_points(summary_df , condition = sample_name, 
                    pct = pct_NANOG, palette = col_NANOG, out_dir = out_dir, 
                    title = "pct_NANOG", y_max = 100)

In [ ]:
wilcox_res <- summary_df %>%
  #filter(sample_name %in% c("D6_RrevG_D", "D6_RrevG_F")) %>%
  #mutate(sample_name = droplevels(sample_name)) %>%
  rstatix::wilcox_test(pct_NANOG ~ sample_name) %>%
  mutate(
    p_use = p,
    stars = case_when(
      p_use < 0.0001 ~ "****",
      p_use < 0.001  ~ "***",
      p_use < 0.01   ~ "**",
      p_use < 0.05   ~ "*",
      TRUE           ~ "ns"
    )
  )

wilcox_res

### F) Intensity per within mcherry +/-

In [ ]:

summary_df <- merged_df %>%
  group_by(image, sample_name, RFP) %>%
  summarise(
    n = n(),
    
    # Calculate simple percentages based on your existing boolean columns
    pct_GATA3 = 100 * mean(GATA3pos, na.rm = TRUE),
    pct_NANOG = 100 * mean(NANOGpos, na.rm = TRUE),
    #pct_mcherry = 100 * mean(mcherrypos, na.rm = TRUE),
    #pct_GFP   = 100 * mean(GFPpos,   na.rm = TRUE),
    
    # Calculate negative cells (None of the markers are positive)
    pct_negative = 100 * mean(!( NANOGpos | GATA3pos), na.rm = TRUE),
    
    # Calculate double positives (Both markers are positive)
    pct_double_GATA3_NANOG = 100 * mean(GATA3pos & NANOGpos, na.rm = TRUE),
    
    .groups = "drop"
  )

head(summary_df)

In [ ]:
title = "doubleneg_RFPgroup_mosaic"
w <- 2.8
h <- 1.6
options(repr.plot.width=w, repr.plot.height=h)

col_rev_group = c("Mosaic_dox"= "#B165A0","Mosaic_none"="#8d8d8dff")

sample_order <- c("Mosaic_none", "Mosaic_dox")

summary_df <- summary_df %>%
mutate(sample_name = factor(sample_name, levels = sample_order))

p = ggplot(summary_df, aes(x = RFP, y =pct_negative , group= sample_name)) +  # dots for each file
    stat_summary( aes(fill = sample_name), 
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.75),
      alpha = 0.6, width = 0.6, fill = "grey80") +   # error bars
    geom_jitter(
      aes(color = sample_name),
      position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
      size = 0.5, alpha = 0.8
    )  +  # average bar
    stat_summary(
      fun.data = mean_se, 
      geom = "errorbar",
      position = position_dodge(width = 0.75),
      width = 0.2)+
    labs(
      title = title,
      y = "%double negative+",
      x = "RFP"
    )+ settheme+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1)
    ) +
      scale_y_continuous(limits = c(0, 110), expand = c(0, 0))+
  scale_color_manual(
    values = col_rev_group,
    name = "dox"
  ) 


ggsave(file.path(out_dir, sprintf("%s.pdf", title)),
                plot = p, width = w, height = h)

p

In [ ]:
summary_df %>%
  #filter(EXP == "EXP84")%>%
  group_by(RFP) %>%
  wilcox_test(
    pct_negative ~ sample_name,
    p.adjust.method = "BH"
  ) 

In [ ]:
title = "doubleneg_RFPgroup_RFP"
w <- 2.8
h <- 1.6
options(repr.plot.width=w, repr.plot.height=h)

col_rev_group = c("Mosaic_dox"= "#B165A0","Mosaic_none"="#8d8d8dff")

sample_order <- c("Mosaic_none", "Mosaic_dox")

summary_df <- summary_df %>%
mutate(sample_name = factor(sample_name, levels = sample_order))

p = ggplot(summary_df, aes(x = RFP, y =pct_negative , group= sample_name)) +  # dots for each file
    stat_summary( aes(fill = sample_name), 
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.75),
      alpha = 0.6, width = 0.6, fill = "grey80") +   # error bars
    geom_jitter(
      aes(color = sample_name),
      position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
      size = 0.5, alpha = 0.8
    )  +  # average bar
    stat_summary(
      fun.data = mean_se, 
      geom = "errorbar",
      position = position_dodge(width = 0.75),
      width = 0.2)+
    labs(
      title = title,
      y = "%double negative+",
      x = "RFP"
    )+ settheme+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1)
    ) +
      scale_y_continuous(limits = c(0, 110), expand = c(0, 0))+
  scale_color_manual(
    values = col_rev_group,
    name = "dox"
  ) 


ggsave(file.path(out_dir, sprintf("%s.pdf", title)),
                plot = p, width = w, height = h)

p

In [ ]:
library(dplyr)
library(tidyr)
library(purrr)

check_test <- function(data, group_var = "sample_name", value_var = "pct_negative",
                        conditions = c("Mosaic_none", "Mosaic_dox"), alpha = 0.05) {

  d <- data %>%
    filter(.data[[group_var]] %in% conditions) %>%
    droplevels()

  d %>%
    group_by(RFP) %>%
    group_modify(~ {
      g <- split(.x[[value_var]], .x[[group_var]])
      g <- g[conditions]                       # keep the two groups in order

      # need at least 3 non-NA points per group for Shapiro
      n_ok <- all(sapply(g, function(x) sum(!is.na(x)) >= 3))

      if (!n_ok) {
        return(tibble(
          n1 = sum(!is.na(g[[1]])), n2 = sum(!is.na(g[[2]])),
          shapiro_p1 = NA_real_, shapiro_p2 = NA_real_,
          levene_p = NA_real_, normal = NA,
          recommended = "too few points (use Wilcoxon / be cautious)"
        ))
      }

      # normality per group
      sp1 <- shapiro.test(g[[1]])$p.value
      sp2 <- shapiro.test(g[[2]])$p.value
      normal <- (sp1 > alpha) & (sp2 > alpha)

      # equal-variance check (F-test; swap for car::leveneTest if preferred)
      var_p <- tryCatch(var.test(g[[1]], g[[2]])$p.value, error = function(e) NA_real_)

      rec <- if (normal) {
        if (!is.na(var_p) && var_p > alpha) "Student t-test (var.equal = TRUE)"
        else "Welch t-test"
      } else {
        "Wilcoxon rank-sum test"
      }

      tibble(
        n1 = sum(!is.na(g[[1]])), n2 = sum(!is.na(g[[2]])),
        shapiro_p1 = sp1, shapiro_p2 = sp2,
        levene_p = var_p, normal = normal,
        recommended = rec
      )
    }) %>%
    ungroup()
}

# usage
check_test(summary_df)